## Correction notice (2026-08-02)

This notebook originally built the red-noise covariance on a **linearly
spaced** frequency grid. That integral does **not converge** for steep
chromatic spectra: J1909-3744's Band 3 chromatic degradation drifts from
51x at 200 grid points to 18x at 6000, and never settles. The grid below
has been changed to **logarithmic** spacing, which converges by ~400
points and agrees with an independent quadrature rule to 0.5%.

This correction **reversed** the array-level band conclusion, so any
numbers recorded from an earlier run of this notebook are void.

See `scripts/11_grid_convergence.py` and `scripts/12_integrator_crosscheck.py`
for the convergence study. The `scripts/` directory is canonical for all
numbers quoted in the write-up; these notebooks are exploratory.


In [ ]:
# 03_build_pulsar_sensitivity_J1909.ipynb
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import hasasia.sensitivity as hsen
from hasasia.sensitivity import red_noise_powerlaw, corr_from_psd
from hasasia.utils import create_design_matrix

yr = 365.25*24*3600.0
df = pd.read_csv("inpta_dr2_full_input_table.csv").set_index("psr")

def build_pulsar(row, sigma_us, cad_per_yr=24, nfreq_spec=300,
                 arn=True, freqs=None):
    """Build a hasasia Pulsar with white noise + (optional) achromatic red
    noise folded into the covariance matrix N. Chromatic noise handled later.
    sigma_us: per-epoch white-noise level (microseconds).
    """
    Tspan = row.span_yr * yr
    N_obs = int(round(row.span_yr * cad_per_yr))
    toas = np.linspace(0, Tspan, N_obs)
    toaerrs = np.ones(N_obs) * sigma_us * 1e-6

    N = np.diag(toaerrs**2)                      # white noise
    if arn and np.isfinite(row.arn_log10A):      # achromatic red noise -> covariance
        rn_freqs = np.logspace(np.log10(1/Tspan), np.log10(cad_per_yr*N_obs/2/Tspan), 800)
        plaw = red_noise_powerlaw(A=10**row.arn_log10A, freqs=rn_freqs,
                                  gamma=row.arn_gamma)
        N = N + corr_from_psd(freqs=rn_freqs, psd=plaw, toas=toas)

    M = create_design_matrix(toas, RADEC=True, PROPER=True, PX=True)
    psr = hsen.Pulsar(toas, toaerrs, phi=row.phi, theta=row.theta,
                      designmatrix=M, N=N)
    if freqs is None:
        freqs = np.logspace(np.log10(1/Tspan), np.log10(1e-7), nfreq_spec)
    spec = hsen.Spectrum(psr, freqs=freqs)
    _ = spec.NcalInv
    return spec

# --- J1909-3744: build with Band 5 white noise, with and without ARN ---
row = df.loc["J1909-3744"]
spec_wn  = build_pulsar(row, sigma_us=row.toa_b5_us, arn=False)
spec_arn = build_pulsar(row, sigma_us=row.toa_b5_us, arn=True)

imin = np.argmin(spec_arn.h_c)
print(f"J1909-3744 (Band 5, white+ARN):")
print(f"  most sensitive h_c = {spec_arn.h_c[imin]:.3e} at {spec_arn.freqs[imin]:.2e} Hz")
print(f"  ARN worsens lowest-f sensitivity by {spec_arn.h_c[0]/spec_wn.h_c[0]:.2f}x")

plt.figure(figsize=(7,5))
plt.loglog(spec_wn.freqs,  spec_wn.h_c,  '--', label='white noise only')
plt.loglog(spec_arn.freqs, spec_arn.h_c, '-',  label='white + achromatic red noise')
plt.xlabel('Frequency [Hz]'); plt.ylabel(r'Characteristic strain $h_c$')
plt.title('J1909$-$3744 single-pulsar sensitivity (InPTA DR2, Band 5)')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig("J1909_single_pulsar_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()